# MRIxFields2026 Task 3 — StarGAN v2 training on Colab

This notebook trains only the organizers' single shared 15-domain Task 3 model. It defaults to **no training**: audit and preprocessing can run first, but pilot/full training cells require both the baseline-milestone acknowledgement and an explicit run switch.

Put this notebook, `task3_colab_train.py`, `task3_pipeline.py`, and `requirements-task3-training-colab.txt` in the same private Drive source folder. Select a GPU runtime before starting. The dataset stays on Drive, but downloads, caches, preprocessing, predictions, and other working files use fast Colab runtime storage under `/content`. Runtime files disappear when the session resets; use the final export cell only for artifacts you need to keep.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, os, shutil, subprocess, sys

PROJECT_DIR_VALUE = '/content/mrixfields_task3'  # @param {type:'string'}
SOURCE_DIR_VALUE = '/content/drive/MyDrive/mrixfields_task3'  # @param {type:'string'}
DATA_DIR_VALUE = '/content/drive/MyDrive/mrixfields2026'  # @param {type:'string'}
SAVE_DIR_VALUE = '/content/drive/MyDrive/mrixfields_task3_saved'  # @param {type:'string'}
PROJECT_DIR = Path(PROJECT_DIR_VALUE)
SOURCE_DIR = Path(SOURCE_DIR_VALUE)
DATA_DIR = Path(DATA_DIR_VALUE)
SAVE_DIR = Path(SAVE_DIR_VALUE)
PREPROCESSED_DIR = PROJECT_DIR / 'preprocessed'
OUTPUT_DIR = PROJECT_DIR / 'training_runs'
OFFICIAL_REPO = Path('/content/MRIxFields2026')
DATA_VIEW_DIR = Path('/content/mrixfields_task3_data_view')
HELPER = PROJECT_DIR / 'task3_colab_train.py'
REQUIREMENTS = PROJECT_DIR / 'requirements-task3-training-colab.txt'
REPORT_DIR = PROJECT_DIR / 'reports'
WEIGHTS_DIR = PROJECT_DIR / 'weights'

for path in (PROJECT_DIR, OUTPUT_DIR, REPORT_DIR, WEIGHTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(PROJECT_DIR / '.cache' / 'torch')
os.environ['HF_HOME'] = str(PROJECT_DIR / '.cache' / 'huggingface')
for filename in ('task3_colab_train.py', 'requirements-task3-training-colab.txt', 'task3_pipeline.py'):
    target = PROJECT_DIR / filename
    candidates = (SOURCE_DIR / filename, Path('/content') / filename)
    source = next((path for path in candidates if path.is_file()), None)
    if source is not None and source.resolve() != target.resolve():
        shutil.copy2(source, target)
for required in (HELPER, REQUIREMENTS):
    if not required.is_file():
        raise FileNotFoundError(
            f'Missing {required.name}. Put it in {SOURCE_DIR} or upload it to /content.'
        )
print('Runtime project:', PROJECT_DIR)
print('Drive source:', SOURCE_DIR)
print('Dataset:', DATA_DIR)
print('Optional saved artifacts:', SAVE_DIR)

## Install only Task 3 training dependencies

Colab already supplies a matching CUDA PyTorch/torchvision build, so the requirements deliberately do not replace it.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU. In Colab choose Runtime > Change runtime type > GPU.')
props = torch.cuda.get_device_properties(0)
print(torch.__version__, torch.version.cuda)
print(props.name, round(props.total_memory / 2**30, 1), 'GiB')

## Fetch the pinned organizer code

The pin was rechecked against live `main` on 2026-07-14. The helper also refuses to train if the checkout differs.

In [ ]:
OFFICIAL_COMMIT = 'f0f4ab2c54ab878b48ddfea8d13f4da512ca411e'
if not (OFFICIAL_REPO / '.git').is_dir():
    if OFFICIAL_REPO.exists():
        raise RuntimeError(f'{OFFICIAL_REPO} exists but is not an organizer Git checkout')
    subprocess.check_call(['git', 'clone', 'https://github.com/MRIxFields/MRIxFields2026.git', str(OFFICIAL_REPO)])
subprocess.check_call(['git', '-C', str(OFFICIAL_REPO), 'fetch', 'origin', OFFICIAL_COMMIT])
subprocess.check_call(['git', '-C', str(OFFICIAL_REPO), 'checkout', '--detach', OFFICIAL_COMMIT])
revision = subprocess.check_output(['git', '-C', str(OFFICIAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert revision == OFFICIAL_COMMIT, revision
print('Official revision:', revision)

## Audit prospective training data

This accepts either organizer casing (`Training_prospective`) or the reported local lowercase casing (`training_prospective`). It checks all 15 modality/field groups, filenames, pairing, shapes, dtypes, and voxel sizes without changing the source data. Add `--check-values` only when you want to read every voxel.

In [ ]:
subprocess.check_call([
    sys.executable, str(HELPER), 'audit-raw',
    '--data-dir', str(DATA_DIR), '--splits', 'pro_train',
    '--report', str(REPORT_DIR / 'pro_train_raw_audit.json'),
])
subprocess.check_call([
    sys.executable, str(HELPER), 'make-data-view',
    '--data-dir', str(DATA_DIR), '--view-dir', str(DATA_VIEW_DIR),
    '--splits', 'pro_train',
])

## Extract the organizer's 220 axial training slices

This is resumable (`--skip_existing`) and writes float32 `[0,1]` NPZ slices to the Colab runtime disk. Prospective extraction is enough for fine-tuning. Retrospective extraction is separate below because it is much larger. The runtime cache is intentionally temporary.

In [ ]:
RUN_PROSPECTIVE_PREPROCESS = False  # @param {type:'boolean'}
if RUN_PROSPECTIVE_PREPROCESS:
    subprocess.check_call([
        sys.executable, str(HELPER), 'preprocess',
        '--official-repo', str(OFFICIAL_REPO), '--data-dir', str(DATA_DIR),
        '--view-dir', str(DATA_VIEW_DIR), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--splits', 'pro_train',
    ])
else:
    print('Prospective preprocessing is OFF. Enable it only after the raw audit passes.')

In [ ]:
RUN_CACHE_AUDIT = False  # @param {type:'boolean'}
if RUN_CACHE_AUDIT:
    subprocess.check_call([
        sys.executable, str(HELPER), 'audit-cache',
        '--preprocessed-dir', str(PREPROCESSED_DIR), '--splits', 'pro_train',
        '--report', str(REPORT_DIR / 'pro_train_cache_audit.json'),
    ])
else:
    print('Cache audit is OFF. Run it after prospective preprocessing.')

## Baseline milestone gate and official initialization checkpoint

The current workspace does not show a validated 180-output baseline archive. Leave the acknowledgement false until the organizer-pretrained validation submission has been built, checked, uploaded by you, and its status reported. By default this copies yesterday's saved checkpoint from Drive to the fast runtime disk, then verifies the pinned official size and SHA-256. It downloads only when the saved-checkpoint option is disabled and the download option is enabled. Loading the checkpoint does not start training.

In [ ]:
BASELINE_MILESTONE_COMPLETE = False  # @param {type:'boolean'}
USE_EXISTING_CHECKPOINT = True  # @param {type:'boolean'}
EXISTING_CHECKPOINT_VALUE = '/content/drive/MyDrive/mrixfields_weights/task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights'  # @param {type:'string'}
DOWNLOAD_OFFICIAL_CHECKPOINT = False  # @param {type:'boolean'}
HF_REVISION = 'f1a7d6b741b64c81871d0856fa2d7295d564c141'
HF_FILE = 'task3_any_to_any_multimodal/stargan_v2/pro_pretrained/weights/checkpoint_epoch50.pth'
EXPECTED_CHECKPOINT_SIZE = 631346455
EXPECTED_CHECKPOINT_SHA256 = 'c2f8172c1fbdbf0426e22c7d3d2014632f175a38cf818f95e72b1917badabdeb'
OFFICIAL_CHECKPOINT = WEIGHTS_DIR / HF_FILE
if USE_EXISTING_CHECKPOINT and DOWNLOAD_OFFICIAL_CHECKPOINT:
    raise ValueError('Enable either the saved checkpoint or download, not both.')
if USE_EXISTING_CHECKPOINT:
    saved_checkpoint = Path(EXISTING_CHECKPOINT_VALUE).expanduser()
    if saved_checkpoint.is_dir():
        saved_checkpoint = saved_checkpoint / 'checkpoint_epoch50.pth'
    if not saved_checkpoint.is_file():
        raise FileNotFoundError(f'Saved checkpoint not found: {saved_checkpoint}')
    if saved_checkpoint.stat().st_size != EXPECTED_CHECKPOINT_SIZE:
        raise RuntimeError(
            f'Checkpoint is incomplete or not the pinned official file: '  
            f'{saved_checkpoint.stat().st_size} bytes; expected {EXPECTED_CHECKPOINT_SIZE}'
        )
    OFFICIAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    if (not OFFICIAL_CHECKPOINT.is_file() or
            OFFICIAL_CHECKPOINT.stat().st_size != saved_checkpoint.stat().st_size):
        print('Copying saved checkpoint to runtime storage...')
        shutil.copy2(saved_checkpoint, OFFICIAL_CHECKPOINT)
    else:
        print('Runtime checkpoint already exists; reusing it.')
elif DOWNLOAD_OFFICIAL_CHECKPOINT:
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(
        repo_id='mrixfields/MRIxFields2026-Baseline', filename=HF_FILE,
        revision=HF_REVISION, local_dir=WEIGHTS_DIR,
    )
    OFFICIAL_CHECKPOINT = Path(downloaded)
elif not OFFICIAL_CHECKPOINT.is_file():
    raise FileNotFoundError('No checkpoint selected or available in runtime storage.')
digest = hashlib.sha256()
with OFFICIAL_CHECKPOINT.open('rb') as checkpoint_file:
    for chunk in iter(lambda: checkpoint_file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
checkpoint_sha256 = digest.hexdigest()
if checkpoint_sha256 != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(
        f'Checkpoint SHA-256 mismatch: {checkpoint_sha256}; expected {EXPECTED_CHECKPOINT_SHA256}'
    )
print('Initialization checkpoint:', OFFICIAL_CHECKPOINT)
print('Size:', OFFICIAL_CHECKPOINT.stat().st_size, 'bytes')
print('SHA-256:', checkpoint_sha256)
print('Training unlocked:', BASELINE_MILESTONE_COMPLETE)

## Pilot: leakage-free subject-held-out prospective training

Run this first after the milestone. `auto` withholds one complete prospective volunteer at subject level and reports bounded **slice-level proxy** nRMSE/SSIM/LPIPS separately for T1W, T2W, and T2FLAIR. The released `pro_pretrained/epoch50` checkpoint has already seen every prospective training volunteer, so it must not initialize an honest holdout. This default pilot starts from scratch. For a stronger honest pilot, initialize from your own retrospective-only checkpoint and attest that it excludes the holdout. These proxies are not public-validation or leaderboard scores.

In [ ]:
RUN_PILOT = False  # @param {type:'boolean'}
PILOT_EPOCHS = 3  # @param {type:'integer'}
PILOT_MICROBATCHES = 100  # @param {type:'integer'}
if RUN_PILOT:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged; training remains locked.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'finetune',
        '--official-repo', str(OFFICIAL_REPO), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR), '--run-name', 'pilot_subject_holdout',
        '--from-scratch', '--holdout-subject', 'auto',
        '--epochs', str(PILOT_EPOCHS), '--max-steps-per-epoch', str(PILOT_MICROBATCHES),
        '--batch-size', '1', '--gradient-accumulation', '1', '--workers', '2',
        '--save-every-steps', '50', '--max-validation-batches', '120',
        '--baseline-milestone-complete',
    ])
else:
    print('Pilot training is OFF.')

### Compare leakage-free pilot epochs by modality

For nRMSE and LPIPS, lower is better; for SSIM, higher is better. These are fixed, balanced slice proxies on the held-out training volunteer, not public-validation scores.

In [ ]:
import json
import pandas as pd
metric_log = OUTPUT_DIR / 'task3_any_to_any_multimodal' / 'stargan_v2' / 'pilot_subject_holdout' / 'train.jsonl'
rows = []
if metric_log.is_file():
    for line in metric_log.read_text().splitlines():
        record = json.loads(line)
        if 'holdout' not in record:
            continue
        groups = {'ALL': record['holdout']['overall'], **record['holdout']['by_modality']}
        for modality, metrics in groups.items():
            rows.append({
                'epoch': record['epoch'], 'modality': modality,
                'nRMSE': metrics['slice_proxy_nrmse'],
                'SSIM': metrics['slice_proxy_ssim'],
                'LPIPS': metrics['slice_proxy_lpips'],
                'samples': metrics['samples'],
            })
    display(pd.DataFrame(rows).sort_values(['epoch', 'modality']))
else:
    print('Run the leakage-free pilot first:', metric_log)

## Continued fine-tuning of the released baseline on all subjects

This is a different experiment from the leakage-free pilot: it continues the released epoch-50 baseline using all prospective training subjects. It has no honest local holdout, so compare it with the released baseline through a complete public-validation submission, not the pilot proxy. Start conservatively (for example one additional epoch), because the initialization is already fine-tuned. Batch 1 with accumulation 8 matches the organizer config's optimization batch of 8 while fitting smaller GPUs. Checkpoints are atomic, written every 500 microbatches, auto-resumed within an epoch, and only the newest three are retained.

In [ ]:
RUN_FULL_FINETUNE = False  # @param {type:'boolean'}
FULL_EPOCHS = 1  # @param {type:'integer'}
MAX_RUNTIME_MINUTES = 300  # @param {type:'integer'}
if RUN_FULL_FINETUNE:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged; training remains locked.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'finetune',
        '--official-repo', str(OFFICIAL_REPO), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR), '--run-name', 'pro_pretrained_all_subjects',
        '--init-checkpoint', str(OFFICIAL_CHECKPOINT), '--holdout-subject', 'none',
        '--epochs', str(FULL_EPOCHS), '--batch-size', '1', '--gradient-accumulation', '8',
        '--workers', '2', '--save-every-steps', '500', '--keep-last', '3',
        '--max-runtime-minutes', str(MAX_RUNTIME_MINUTES),
        '--baseline-milestone-complete',
    ])
else:
    print('Full fine-tuning is OFF.')

## Public-validation inference for one checkpoint

Keep `task3_pipeline.py` in `SOURCE_DIR` (or upload it to `/content`); the setup cell copies it into the runtime project. The released validation targets are private, so this produces and validates all 180 predictions but cannot calculate local validation metrics. Package the result with the current official builder and upload it yourself to obtain leaderboard metrics. Use the same seed for every checkpoint comparison. Run the smoke first, then the full three-modality sweep.

In [ ]:
PIPELINE = PROJECT_DIR / 'task3_pipeline.py'
VALIDATION_DIR_VALUE = '/content/drive/MyDrive/mrixfields2026/validation_prospective'  # @param {type:'string'}
VALIDATION_CHECKPOINT_VALUE = ''  # @param {type:'string'}
SWEEP_MODE = 'pro_pretrained'  # @param {type:'string'}
SWEEP_EPOCH = 'epoch50'  # @param {type:'string'}
RUN_VALIDATION_SMOKE = False  # @param {type:'boolean'}
RUN_FULL_VALIDATION = False  # @param {type:'boolean'}
VALIDATION_DIR = Path(VALIDATION_DIR_VALUE)
VALIDATION_CHECKPOINT = Path(VALIDATION_CHECKPOINT_VALUE) if VALIDATION_CHECKPOINT_VALUE else OFFICIAL_CHECKPOINT
VALIDATION_OUTPUT = PROJECT_DIR / 'validation_predictions'
base_command = [
    sys.executable, str(PIPELINE), 'infer', '--input-root', str(VALIDATION_DIR),
    '--output-dir', str(VALIDATION_OUTPUT), '--official-repo', str(OFFICIAL_REPO),
    '--checkpoint', str(VALIDATION_CHECKPOINT), '--sweep-mode', SWEEP_MODE,
    '--sweep-epoch', SWEEP_EPOCH, '--device', 'cuda', '--batch-size', '1',
    '--seed', '20260712',
]
if RUN_VALIDATION_SMOKE or RUN_FULL_VALIDATION:
    if not PIPELINE.is_file():
        raise FileNotFoundError(f'Upload the prepared validation runner: {PIPELINE}')
if RUN_VALIDATION_SMOKE:
    subprocess.check_call(base_command + ['--modalities', 'T1W', '--smoke', '--resume'])
if RUN_FULL_VALIDATION:
    subprocess.check_call(base_command + ['--modalities', 'T1W', 'T2W', 'T2FLAIR', '--resume'])
    subprocess.check_call([
        sys.executable, str(PIPELINE), 'validate-outputs',
        '--input-root', str(VALIDATION_DIR), '--output-dir', str(VALIDATION_OUTPUT),
        '--modalities', 'T1W', 'T2W', 'T2FLAIR',
        '--sweep-mode', SWEEP_MODE, '--sweep-epoch', SWEEP_EPOCH,
        '--report', str(REPORT_DIR / f'validation_{SWEEP_MODE}_{SWEEP_EPOCH}.json'),
    ])
if not RUN_VALIDATION_SMOKE and not RUN_FULL_VALIDATION:
    print('Validation inference is OFF.')

## Optional: retrospective pretraining from scratch

This is not the recommended first Colab experiment. It requires the much larger retrospective cache and the official 200,000-step schedule. Use the pretrained initialization above unless you have a concrete, measured reason to pay this compute cost.

In [ ]:
RUN_RETROSPECTIVE_PREPROCESS = False  # @param {type:'boolean'}
if RUN_RETROSPECTIVE_PREPROCESS:
    subprocess.check_call([
        sys.executable, str(HELPER), 'preprocess',
        '--official-repo', str(OFFICIAL_REPO), '--data-dir', str(DATA_DIR),
        '--view-dir', str(DATA_VIEW_DIR), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--splits', 'retro_train',
    ])
else:
    print('Retrospective preprocessing is OFF.')

In [ ]:
RUN_RETROSPECTIVE_PRETRAIN = False  # @param {type:'boolean'}
PRETRAIN_STEPS = 200000  # @param {type:'integer'}
if RUN_RETROSPECTIVE_PRETRAIN:
    if not BASELINE_MILESTONE_COMPLETE:
        raise RuntimeError('Baseline milestone is not acknowledged; training remains locked.')
    subprocess.check_call([
        sys.executable, str(HELPER), 'pretrain',
        '--official-repo', str(OFFICIAL_REPO), '--preprocessed-dir', str(PREPROCESSED_DIR),
        '--output-dir', str(OUTPUT_DIR), '--run-name', 'retro_scratch_colab',
        '--total-steps', str(PRETRAIN_STEPS), '--batch-size', '1', '--workers', '2',
        '--save-every', '1000', '--keep-last', '3', '--max-runtime-minutes', '300',
        '--baseline-milestone-complete',
    ])
else:
    print('Retrospective pretraining is OFF.')

## Save only selected final artifacts to Drive

Everything under `PROJECT_DIR` is temporary and disappears when the Colab runtime resets. This opt-in cell copies the small JSON reports plus one explicitly selected checkpoint and/or archive. It does not copy caches, extracted slices, full prediction trees, or all training checkpoints. Leave paths blank for items you do not want to save.

In [ ]:
SAVE_SELECTED_ARTIFACTS = False  # @param {type:'boolean'}
CHECKPOINT_TO_SAVE_VALUE = ''  # @param {type:'string'}
ARCHIVE_TO_SAVE_VALUE = ''  # @param {type:'string'}
if SAVE_SELECTED_ARTIFACTS:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    saved = []
    if REPORT_DIR.is_dir():
        report_destination = SAVE_DIR / 'reports'
        shutil.copytree(REPORT_DIR, report_destination, dirs_exist_ok=True)
        saved.append(report_destination)
    for value, label in (
        (CHECKPOINT_TO_SAVE_VALUE, 'checkpoint'),
        (ARCHIVE_TO_SAVE_VALUE, 'archive'),
    ):
        if not value.strip():
            continue
        source = Path(value).expanduser()
        if not source.is_file():
            raise FileNotFoundError(f'Selected {label} does not exist: {source}')
        destination = SAVE_DIR / source.name
        shutil.copy2(source, destination)
        saved.append(destination)
    print('Saved artifacts:')
    for path in saved:
        print(' -', path)
else:
    print('Artifact export is OFF; runtime files remain only under', PROJECT_DIR)